In [2]:
import requests
import json

url = "https://mast.stsci.edu/api/v0/invoke"

query = {
    "service": "Mast.Caom.Filtered",
    "params": {
        "columns": "*",
        "filters": [
            {"paramName": "obs_collection", "values": ["JWST"]},
            {"paramName": "instrument_name", "values": ["NIRCAM", "MIRI"]},
            {"paramName": "dataproduct_type", "values": ["image"]}
        ]
    }
}

response = requests.post(url, data={"request": json.dumps(query)})
obs = response.json()["_uncal.fits"]

# Take one obsid
obsid = obs[0]["obsid"]

# Now fetch products
query2 = {
    "service": "Mast.Caom.Products",
    "params": {
        "obsid": obsid,
        "columns": "*"
    }
}

response2 = requests.post(url, data={"request": json.dumps(query2)})
products = response2.json()["data"]

# Filter raw files
uncal = [p for p in products if p["calib_level"] == 1]

print(uncal[:3])

KeyError: '_uncal.fits'

In [4]:
import requests
import json

url = "https://mast.stsci.edu/api/v0/invoke"

query = {
    "service": "Mast.Caom.Filtered",
    "params": {
        "columns": "*",
        "filters": [
            {"paramName": "obs_collection", "values": ["JWST"]},
            {"paramName": "instrument_name", "values": ["NIRCAM", "MIRI"]},
            {"paramName": "dataproduct_type", "values": ["image"]}
        ]
    },
    "format": "json"
}

response = requests.post(url, data={"request": json.dumps(query)})
result = response.json()

# 🔥 SAFE ACCESS
if "data" not in result:
    print("❌ API Error:", result)
else:
    obs = result["data"]
    print(f"✅ Found {len(obs)} observations")


def safe_get_data(response):
    result = response.json()
    
    if "status" in result and result["status"] == "error":
        raise Exception(f"MAST API Error: {result}")
    
    if "data" not in result:
        raise Exception(f"Unexpected response: {result}")
    
    return result["data"]

✅ Found 0 observations


In [7]:
import requests
import json

url = "https://mast.stsci.edu/api/v0/invoke"

query = {
    "service": "Mast.Caom.Filtered",
    "params": {
        "columns": "*",
        "filters": [
            {"paramName": "obs_collection", "values": ["JWST"]},
            {"paramName": "instrument_name", "values": ["NIRCam", "MIRI"]}
        ]
    },
    "format": "json"
}

query2 = {
    "service": "Mast.Caom.Products",
    "params": {
        "obsid": obsid,
        "columns": "*"
    },
    "format": "json"
}


response = requests.post(url, data={"request": json.dumps(query)})
result = response.json()

# 🔥 SAFE ACCESS
if "data" not in result:
    print("❌ API Error:", result)
else:
    obs = result["data"]
    print(f"✅ Found {len(obs)} observations")


def safe_get_data(response):
    result = response.json()
    
    if "status" in result and result["status"] == "error":
        raise Exception(f"MAST API Error: {result}")
    
    if "data" not in result:
        raise Exception(f"Unexpected response: {result}")
    
    return result["data"]

NameError: name 'obsid' is not defined

In [1]:
import requests
import json

MAST_URL = "https://mast.stsci.edu/api/v0/invoke"


def mast_query(payload):
    response = requests.post(MAST_URL, data={"request": json.dumps(payload)})
    result = response.json()

    # Robust error handling
    if "status" in result and result["status"] == "error":
        raise Exception(f"MAST API Error: {result}")

    if "data" not in result:
        raise Exception(f"Unexpected response: {result}")

    return result["data"]


# -------------------------------
# STEP 1: Get JWST observations
# -------------------------------
obs_query = {
    "service": "Mast.Caom.Filtered",
    "params": {
        "columns": "obsid,instrument_name,target_name",
        "filters": [
            {"paramName": "obs_collection", "values": ["JWST"]}
        ]
    },
    "format": "json"
}

obs_data = mast_query(obs_query)

print(f"Total JWST observations: {len(obs_data)}")

# -------------------------------
# STEP 2: Filter instruments (correct names!)
# -------------------------------
filtered_obs = [
    o for o in obs_data
    if o["instrument_name"] in ["NIRCam", "MIRI"]
]

print(f"NIRCam + MIRI observations: {len(filtered_obs)}")

# Take a few obsids (avoid huge downloads)
obsids = [o["obsid"] for o in filtered_obs[:3]]

# -------------------------------
# STEP 3: Fetch products
# -------------------------------
all_uncal = []

for obsid in obsids:
    prod_query = {
        "service": "Mast.Caom.Products",
        "params": {
            "obsid": obsid,
            "columns": "*"
        },
        "format": "json"
    }

    products = mast_query(prod_query)

    # -------------------------------
    # STEP 4: Filter RAW (uncalibrated)
    # -------------------------------
    uncal = [
        p for p in products
        if p.get("calib_level") == 1
    ]

    print(f"obsid {obsid} → {len(uncal)} uncal files")

    all_uncal.extend(uncal)

# -------------------------------
# STEP 5: Show download URLs
# -------------------------------
print("\nSample uncal files:\n")

for f in all_uncal[:5]:
    print(f["dataURI"])

Total JWST observations: 408808
NIRCam + MIRI observations: 0

Sample uncal files:

